# 05 — Strahler diagnostics & self-similarity

Companion to [`examples/plot_strahler.py`](../examples/plot_strahler.py).

The headline finding of Eloy et al. 2017 is that trees grown under
mechanical + light constraints aren't just shaped like real trees —
they are **statistically self-similar** in the same way real trees are.
This notebook grows a mature tree and reads off the three classical
diagnostics:

1. **Horton–Strahler scaling.** Branch count, length, and area each
   shrink geometrically with Strahler order — straight lines on a
   semilog plot.
2. **Leonardo's rule.** At every junction, the daughter cross-section
   areas sum to the parent's. Histogramming the ratio peaks at 1.
3. **Tokunaga matrix.** Counts of side-branchings as a function of
   order difference; diagonal-dominant in a self-similar tree.

All three live in `mechatree.stats` as pure-Python helpers operating on
a `PyTree`.

In [ ]:
from pathlib import Path

from mechatree.config import load_config
from mechatree.plotting import plot_strahler_diagnostics
from mechatree.simulate import grow_tree
from mechatree.stats import leonardo_ratios, strahler_summary, tokunaga_matrix

cfg = load_config(Path("../examples/forest.yaml").resolve())

## Grow a mature tree

We need enough generations for the upper Strahler orders to populate.
200 generations is a reasonable middle ground; the example script uses
300 for the published-quality figure.

In [ ]:
tree = grow_tree(cfg, n_generations=200, seed=42)
summary = strahler_summary(tree)
print(f"Tree: {tree.get_number_of_branches()} branches, Strahler max order = {summary.max_order}")

## Branch counts, lengths, areas per order

`strahler_summary` returns mean branch length, mean diameter and mean
cross-section area per Strahler order. The dimensionless ratios across
consecutive orders are the *bifurcation ratio* (count), *length ratio*
and *area ratio* from the paper.

In [ ]:
print(f"{'order':>6}  {'n':>6}  {'<L>':>8}  {'<d>':>8}  {'<A>':>10}")
for k in range(summary.max_order):
    print(
        f"{k + 1:>6}  {summary.n_branches[k]:>6}  "
        f"{summary.mean_length[k]:>8.3f}  "
        f"{summary.mean_diameter[k]:>8.4f}  "
        f"{summary.mean_area[k]:>10.5f}"
    )

## Leonardo's rule

At a binary junction, the rule says
$A_\text{parent} \approx A_\text{daughter,1} + A_\text{daughter,2}$.
`leonardo_ratios` computes
$\frac{A_1 + A_2}{A_\text{parent}}$ at every junction; a self-similar
tree gives a tight distribution around 1.

In [ ]:
ratios = leonardo_ratios(tree)
if ratios.size:
    print(f"Leonardo ratio at {ratios.size} junctions:")
    print(
        f"  mean   = {ratios.mean():.3f}\n"
        f"  std    = {ratios.std():.3f}\n"
        f"  median = {float(sorted(ratios)[ratios.size // 2]):.3f}"
    )

## Tokunaga matrix

$T_{ij}$ is the average number of order-$i$ branches that side-branch
off an order-$j$ trunk. A self-similar tree has $T_{ij} = a c^{j-i-1}$
(Tokunaga's law). Diagonal entries here represent main-stem
continuation; off-diagonals are side branches.

In [ ]:
T = tokunaga_matrix(tree)
print(f"Tokunaga matrix ({T.shape[0]}x{T.shape[1]}):")
print(T)

## Putting it together

`plot_strahler_diagnostics` rolls these into one 2×2 figure: branch
count, mean length and mean area per order (semilog), plus the
Leonardo histogram. The dashed reference lines mark the slopes
predicted by the Eloy 2017 model.

In [ ]:
fig = plot_strahler_diagnostics(tree)
fig.show()

## Where to next

- Re-run this notebook with one of the neural genomes from
  [03_neural_genome.ipynb](03_neural_genome.ipynb) and compare slopes.
- Drive a [forest](02_forest_under_wind.ipynb), pick a survivor, and
  see whether competition shifts the self-similarity exponents.
- Read Eloy et al. 2017, fig. 5–6, for the empirical comparison
  against real-tree data.